In [1]:
import pandas as pd

In [2]:
def read_tbl():
    df = pd.read_csv("../data/input/2025-8-26/Officer List 082225.csv")
    df = df.fillna("")
    df = df.rename(columns={"Agency": "agency_name",
                            "Agency ORI": "agency_ori",
                            "Last_Name": "last_name",
                            "First_Name": "first_name",
                            "Middle_Name": "middle_name",
                            "AppointedDate": "start_date",
                            "TerminationDate": "end_date",
                            "Term_Desc": "separation_reason", 
                            "PostId": "uid"})
    
    df = df.drop(columns=["Rank" ,"CertStatus_atTerm", "CertStatus_Current", "FinalActions"])
    return df 

def clean_sep_reason(df):
    df.loc[:, "separation_reason"] = (df.separation_reason
                                      .str.lower()
                                      .str.strip()
                                      .str.replace(r"misconduct - no", "not for misconduct", regex=True)
                                      .str.replace(r"misconduct - yes", "misconduct", regex=True)
    )
    
    return df 


def clean_agency_name(df):
    df.loc[:, "agency_name"] = (df
                                .agency_name
                                .str.lower()
                                .str.strip()
                                .str.replace(r"^az ", "arizona ", regex=True)
                                .str.replace(r" dept ", " department ", regex=False)
                                .str.replace(r"dept$", "department", regex=True)
                                .str.replace(r"departme$", "department", regex=True)
                                .str.replace(r" enf ", " enforcement ", regex=False)
                                .str.replace(r" az ", " arizona ", regex=False)
                                .str.replace(r" & ", " and ", regex=False)
                                .str.replace(r"contr$", "control", regex=True)
                                .str.replace(r"pd$", "police department", regex=True)
                                .str.replace(r"departm$", "department", regex=True)
                                .str.replace(r" cty ", " county ", regex=False)
                                .str.replace(r"-(\w+)$", r"- \1", regex=True)
                                .str.replace(r"(\w+)\,(\w+)", r"\1, \2", regex=True)
                                .str.replace(r"animal se", "animal services", regex=False)
                                .str.replace(r"^ret\,? ", "", regex=True)
                                .str.replace(r"sheriffs", "sheriff's", regex=False)
                                .str.replace(r"comm coll", "community college", regex=False)
    )
    return df

def read_ori():
    df = pd.read_csv("../data/input/da35158-0001.csv")

    df = df[["ORI9", "NAME","COUNTYNAME", "UANAME","AGCYTYPE", "LG_NAME", "ADDRESS_NAME", "ADDRESS_STR1", "ADDRESS_CITY",  "ADDRESS_ZIP", "LG_POPULATION", "INTPTLAT", "INTPTLONG"]] 	

    df = df.rename(columns={"ORI9": "agency_ori"})								
    return df 


dfa = read_tbl()

dfa = dfa.pipe(clean_sep_reason).pipe(clean_agency_name)

dfa

,agency_name,agency_ori,uid,last_name,first_name,middle_name,start_date,end_date,separation_reason
0,arizona drug control district,,17032,Pennington,Carroll,A,,4/22/81,
1,pinal county sheriff's office,,21252,Escobedo,Ricardo,O,,3/19/84,
2,safford police department,AZ0050300,11978,Golding,G,A,7/1/32,6/30/72,other/unknown
3,maricopa county sheriff's office,AZ0070000,17586,Baldwin,Clarenc,E,10/22/33,12/29/72,other/unknown
4,union pacific railroad police department,AZ007379E,24104,Macqueen,Herbert,C,4/3/37,3/1/73,other/unknown
...,...,...,...,...,...,...,...,...,...
70641,arizona department of public safety,AZ0079900,45799,Pierce,Seth,Alan,8/21/25,,active
70642,camp verde marshals office,AZ0131300,52635,Bitterman,Jason,Patrick,8/21/25,,active
70643,goodyear police department,AZ0071500,73422,Schexnyder,Mackayla,Lee,8/21/25,,active
70644,tucson police department,AZ0100300,63468,Frias,Carlos,Gerardo,8/21/25,,active


In [3]:
dfb = read_ori()

df = pd.merge(dfa, dfb, on="agency_ori")

df

,agency_name,agency_ori,uid,last_name,first_name,middle_name,start_date,end_date,separation_reason,NAME,...,UANAME,AGCYTYPE,LG_NAME,ADDRESS_NAME,ADDRESS_STR1,ADDRESS_CITY,ADDRESS_ZIP,LG_POPULATION,INTPTLAT,INTPTLONG
0,safford police department,AZ0050300,11978,Golding,G,A,7/1/32,6/30/72,other/unknown,SAFFORD POLICE DEPARTMENT,...,"Safford, AZ Urban Cluster",(000) Local police department,SAFFORD CITY,SAFFORD POLICE DEPARTMENT,525 10TH AVENUE,SAFFORD,85546.0,9566,32.931828,-109.878310
1,maricopa county sheriff's office,AZ0070000,17586,Baldwin,Clarenc,E,10/22/33,12/29/72,other/unknown,MARICOPA COUNTY SHERIFF'S OFFICE,...,_Not a Census place,(001) Sheriff's office,MARICOPA COUNTY,MARICOPA COUNTY SHERIFF'S OFFICE,19TH FLOOR,PHOENIX,85003.0,3817117,33.346541,-112.495534
2,arizona department of agriculture - animal ser...,AZLSB0000,10902,Mounce,L,J,10/15/38,7/19/61,other/unknown,AZ DEPT AGRICULTURE ANIMAL SERVICES DIV,...,"Phoenix--Mesa, AZ Urbanized Area",(006) Special jurisdiction,State of Arizona,AZ DEPT OF AGRICULTURE ANIMAL SERVICES DIVISION,NaN,PHOENIX,NaN,888888888,33.346541,-112.495534
3,arizona department of public safety,AZ0079900,16567,Rogers,Earl,B,1/6/41,4/17/73,other/unknown,ARIZONA DEPT OF PUBLIC SAFETY HQ PHOENIX,...,"Phoenix--Mesa, AZ Urbanized Area",(005) State law enforcement agency,State of Arizona,ARIZONA DEPARTMENT OF PUBLIC SAFETY COMMUNICAT...,NaN,PHOENIX,NaN,888888888,33.346541,-112.495534
4,maricopa county sheriff's office,AZ0070000,9282,Baker,V,E,8/13/41,6/22/73,other/unknown,MARICOPA COUNTY SHERIFF'S OFFICE,...,_Not a Census place,(001) Sheriff's office,MARICOPA COUNTY,MARICOPA COUNTY SHERIFF'S OFFICE,19TH FLOOR,PHOENIX,85003.0,3817117,33.346541,-112.495534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68223,arizona department of public safety,AZ0079900,45799,Pierce,Seth,Alan,8/21/25,,active,ARIZONA DEPT OF PUBLIC SAFETY HQ PHOENIX,...,"Phoenix--Mesa, AZ Urbanized Area",(005) State law enforcement agency,State of Arizona,ARIZONA DEPARTMENT OF PUBLIC SAFETY COMMUNICAT...,NaN,PHOENIX,NaN,888888888,33.346541,-112.495534
68224,camp verde marshals office,AZ0131300,52635,Bitterman,Jason,Patrick,8/21/25,,active,CAMP VERDE MARSHAL'S OFFICE,...,"Camp Verde, AZ Urban Cluster",(000) Local police department,CAMP VERDE TOWN,CAMP VERDE MARSHAL'S OFFICE,646 S. FIRST STREET,CAMP VERDE,86322.0,10873,34.630044,-112.573745
68225,goodyear police department,AZ0071500,73422,Schexnyder,Mackayla,Lee,8/21/25,,active,GOODYEAR POLICE DEPARTMENT,...,"Avondale--Goodyear, AZ Urbanized Area",(000) Local police department,GOODYEAR CITY,GOODYEAR POLICE DEPARTMENT,1111 S LITCHFIELD ROAD,GOODYEAR,85338.0,65275,33.346541,-112.495534
68226,tucson police department,AZ0100300,63468,Frias,Carlos,Gerardo,8/21/25,,active,TUCSON POLICE DEPARTMENT,...,"Tucson, AZ Urbanized Area",(000) Local police department,TUCSON CITY,TUCSON POLICE DEPARTMENT,270 SOUTH STONE STREET,TUCSON,85701.0,520116,32.128238,-111.783018


In [4]:
df.head(10).to_csv("../data/output/arizona_head.csv", index=False)